# Voice-to-Voice AI Agent using OpenAI

- This project builds a Voice-to-Voice AI Assistant
- It takes microphone input → processes it → generates AI response → plays audio output
- Uses:
    - sounddevice (for audio recording & playback)
    - OpenAI Agents (for AI reasoning)
    - Voice Pipeline (for speech-to-speech interaction)

**Step 1: Detect Audio Devices**

Detects:
🎤 Microphone (input device)
🔊 Speaker (output device)
Used for recording and playback

In [ ]:
import sounddevice as sd 

input_device = sd.query_devices(kind= "input")
output_device = sd.query_devices(kind= "output")

input_device,output_device
    

**Step 2: Record Audio Input**

Explanation:
- Records audio in real-time chunks
- Stores chunks in recorded_chunks
- Each chunk = small piece of audio data

In [ ]:
import numpy as np

recorded_chunks = []

with sd.InputStream(
    samplerate=int(input_device['default_samplerate']),
    channels=1,
    dtype=np.int16,
    callback=lambda indata, frames, time, status: recorded_chunks.append(indata.copy())
):
    input()

**Step 3: Inspect Recorded Data**

Explanation:
- Records audio in real-time chunks
- Stores chunks in recorded_chunks
- Each chunk = small piece of audio data

In [ ]:
len(recorded_chunks)

In [ ]:
recorded_chunks[0].shape, recorded_chunks[1].shape, recorded_chunks[2].shape

**Step 4: Combine Audio Chunks**

Explanation:
Checks:
- Number of chunks recorded
- Shape of audio data

In [ ]:
import numpy as np

audio_buffer = np.concatenate(recorded_chunks)
audio_buffer.shape

**Step 5: Play Recorded Audio**

Explanation:
- Plays back recorded voice
- Useful for testing recording and playback functionality

In [ ]:
sd.play(audio_buffer, samplerate=int(output_device['default_samplerate']))
sd.wait()

**Step 6: Convert Audio into Model Input**

Explanation:
- Converts raw audio → structured input
- Required for OpenAI voice pipeline

In [ ]:
from agents.voice import AudioInput

audio_input = AudioInput(
    buffer = audio_buffer,
    frame_rate=int(input_device['default_samplerate']),
    channels = audio_buffer.shape[1],
)
audio_input

**Step 7: Create AI Agent**

Explanation:
- Creates AI agent
- Uses:
- Name
- Instructions (behavior of assistant)

In [ ]:
import os
from agents import Agent,Runner
os.environ["OPENAI_API_KEY"] = "Your_api_key_here"
agent = Agent(
    name = "Assistant",
    instructions = (
        """Repeat the user's question back to them, and answer it. Note that the user is speaking with you to you via voice interface,"""
        """ although you are reading and writing text to respond.Nonetheless, ensure that your written response is easily translatable to voice."""
    ),
    model = "gpt-4.1-nano"
)

**Step 8: Create Voice Workflow**

Explanation:
- Connects: Audio input → AI agent → Audio output

In [ ]:
from agents.voice import SingleAgentVoiceWorkflow
workflow = SingleAgentVoiceWorkflow(agent)

**Step 9: Configure Voice Settings**

Explanation:
- Connects: Audio input → AI agent → Audio output

In [ ]:
from agents.voice import TTSModelSettings, VoicePipelineConfig

custom_tts_settings = TTSModelSettings(
    instructions=(
        "Personality: upbeat, friendly, persuasive guide.\n"
        "Tone: Friendly, clear, and reassuring, creating a calm atmosphere and making "
        "the listener feel confident and comfortable.\n"
        "Pronunciation: Clear, articulate, and steady, ensuring each instruction is "
        "easily understood while maintaining a natural, conversational flow.\n"
        "Tempo: Speak relatively fast, include brief pauses and after before questions.\n"
        "Emotion: Warm and supportive, conveying empathy and care, ensuring the listener "
        "feels guided and safe throughout the journey."
    )
)
voice_pipeline_config = VoicePipelineConfig(tts_settings = custom_tts_settings)

**Step 10: Create Voice Pipeline**

Explanation:
- Main engine of system
    Handles:
    - Speech → Text
    - AI processing
    - Text → Speech

In [ ]:
from agents.voice import VoicePipeline
pipeline = VoicePipeline(workflow=workflow, config = voice_pipeline_config)

**Step 11: Run the Pipeline**

Explanation:
- Sends audio to AI
- Receives streamed response
- Collects audio output chunks

In [ ]:
result = await pipeline.run(audio_input=audio_input)

response_chunks = []

async for event in result.stream():
    if event.type == "voice_stream_event_audio":
        response_chunks.append(event.data)

# concatenate all of the chunks into a single audio buffer
response_audio_buffer = np.concatenate(response_chunks, axis=0)

# openai sample rate is 24000
openai_sample_rate = 24_000

# play the response
sd.play(response_audio_buffer, samplerate=openai_sample_rate)
sd.wait()

**Step 12: Continuous Voice Assistant**

Explanation:
- Runs assistant in loop
    User can:
    - Speak continuously
    - Exit anytime

In [ ]:
async def voice_assistant_optimized():
    while True:
        # check for input to either provide voice or exit
        cmd = input("Press Enter to speak (or type 'q' to exit): ")
        if cmd.lower() == "q":
            print("Exiting...")
            break
        print("Listening...")
        recorded_chunks = []

         # start streaming from microphone until Enter is pressed
        with sd.InputStream(
            samplerate=int(input_device['default_samplerate']),
            channels=1,
            dtype='int16',
            callback=lambda indata, frames, time, status: recorded_chunks.append(indata.copy())
        ):
            input()

        # concatenate chunks into single buffer
        recording = np.concatenate(recorded_chunks, axis=0)

        # input the buffer and await the result
        audio_input = AudioInput(buffer=recording)

        result = await pipeline.run(audio_input)

         # transfer the streamed result into chunks of audio
        response_chunks = []
        async for event in result.stream():
            if event.type == "voice_stream_event_audio":
                response_chunks.append(event.data)

        response_audio_buffer = np.concatenate(response_chunks, axis=0)

        # play response
        print("Assistant is responding...")
        sd.play(response_audio_buffer, samplerate=openai_sample_rate)
        sd.wait()
        print("---")

# run the voice assistant
await voice_assistant_optimized()

**#Summary#**

- This project builds a real-time Voice AI Assistant
- Converts speech → AI response → speech
- Uses OpenAI Agents + Audio streaming
- Demonstrates:
    - Speech Processing
    - AI Interaction
    - Real-time pipelines